# Process Long Subtitles — 語意斷句強化版

將 AI 產生（如 Rask.ai、Whisper）的過長 SRT 字幕，依**語意邊界**拆分成多行，並自動均分時間軌，提升字幕可讀性。

斷句優先順序：
1. 強標點（`. ! ? ; :`）之後
2. 弱標點（`,` `—` `-`）之後
3. 連接詞（and, but, because, which, that …）之前
4. 以上皆無 → 在空白處硬切（不切斷英文單字）

每段長度盡量不超過 `max_length`，時間戳依段數比例分配。

In [ ]:
!pip install pysrt

In [ ]:
import re
import pysrt

In [ ]:
# 連接詞：在這些詞「之前」斷句較自然
CONJUNCTIONS = [
    "and", "but", "or", "nor", "so", "yet", "for",
    "because", "although", "though", "while", "whereas",
    "which", "that", "who", "when", "where", "if", "unless",
    "since", "after", "before", "as",
]

def _best_break(text, max_length):
    """在 <= max_length 的範圍內，找最靠後的語意斷點，回傳切割位置 (index)。
    找不到語意斷點時回傳 None，交給呼叫端 fallback。"""
    window = text[:max_length]

    # 1) 強標點（句子結束）：. ! ? ; :  —— 在標點之後切
    strong = list(re.finditer(r'[.!?;:]+\s', window))
    if strong:
        return strong[-1].end()

    # 2) 弱標點：逗號、破折號 —— 在標點之後切
    weak = list(re.finditer(r'[,\u2014-]+\s', window))
    if weak:
        return weak[-1].end()

    # 3) 連接詞之前切（取範圍內最靠後、且不在開頭的連接詞）
    best = None
    for m in re.finditer(r'\b(' + '|'.join(CONJUNCTIONS) + r')\b', window, re.IGNORECASE):
        if m.start() > 0:  # 不要切在最前面，否則沒進展
            best = m.start()
    if best:
        return best

    # 4) 找不到語意斷點
    return None

def split_line_semantic(line, max_length):
    """把一行依語意邊界切成多段，每段長度盡量 <= max_length。"""
    line = line.strip()
    if len(line) <= max_length:
        return [line]

    parts = []
    remaining = line
    while len(remaining) > max_length:
        pos = _best_break(remaining, max_length)
        if pos is None:
            # fallback：在最後一個空白處硬切，避免切斷英文單字
            cut = remaining.rfind(' ', 0, max_length)
            pos = cut if cut > 0 else max_length
        chunk = remaining[:pos].strip()
        if chunk:
            parts.append(chunk)
        remaining = remaining[pos:].strip()
    if remaining:
        parts.append(remaining)
    return parts

In [ ]:
def process_subtitles(subs, max_length):
    processed_subs = []
    k = 1
    for sub in subs:
        lines = sub.text.split('\n')
        new_subs = []
        for line in lines:
            new_subs.extend(split_line_semantic(line, max_length))
        # 用總毫秒數做時間切分，避免跨分鐘時 .seconds 丟失分鐘的問題
        span_ms = sub.end.ordinal - sub.start.ordinal
        n = len(new_subs)
        for j, text in enumerate(new_subs):
            new_sub = pysrt.SubRipItem(index=k, text=text)
            if n == 1:
                new_sub.start = sub.start
                new_sub.end = sub.end
            else:
                new_sub.start = pysrt.SubRipTime.from_ordinal(sub.start.ordinal + span_ms * j // n)
                new_sub.end = pysrt.SubRipTime.from_ordinal(sub.start.ordinal + span_ms * (j + 1) // n)
            processed_subs.append(new_sub)
            k += 1
    return processed_subs

In [ ]:
def process_file(input_file, output_file, max_length=100):
    subs = pysrt.open(input_file)
    processed_subs = process_subtitles(subs, max_length)
    with open(output_file, 'w', encoding='utf-8') as f:
        for sub in processed_subs:
            f.write(str(sub))
            f.write('\n')
    print(f'完成：{output_file}（共 {len(processed_subs)} 句）')

## 使用範例

In [ ]:
# Usage
process_file('video_en.srt', 'video_en_new.srt', max_length=100)